# Performance characteristics

This page measures the taxonomy layer rather than describing it. Every number
below is produced when the documentation is built, so the claims cannot drift
away from the code.

The committed benchmark suite (`pixi run bench`) uses `pytest-benchmark` and is
the authority for regression tracking; this page exists to show the *shape* of
the cost curves.

In [ ]:
import time

import numpy as np

from summer4 import Property, PropertyMap


def timed(fn, repeats=5):
    """Return the best wall-clock time of ``fn`` over ``repeats`` runs."""
    best = float("inf")
    for _ in range(repeats):
        start = time.perf_counter()
        fn()
        best = min(best, time.perf_counter() - start)
    return best


def cartesian(factors):
    props = [Property(f"p{i}", tuple(f"t{j}" for j in range(n))) for i, n in enumerate(factors)]
    pmap = PropertyMap.from_property(props[0])
    for prop in props[1:]:
        pmap = pmap.stratify(prop)
    return pmap

## Building a map

Construction is a gather plus a concatenate per stratification, so cost is
linear in the size of the **output** table, not in the number of axes.

In [ ]:
shapes = [(10,), (10, 10), (10, 10, 10), (10, 10, 10, 10), (10, 10, 10, 10, 10)]

print(f"{'compartments':>13}  {'axes':>4}  {'build (ms)':>11}  {'us/compartment':>14}")
build_rows = []
for factors in shapes:
    size = int(np.prod(factors))
    seconds = timed(lambda f=factors: cartesian(f))
    build_rows.append((size, seconds))
    print(f"{size:>13,}  {len(factors):>4}  {seconds * 1e3:>11.3f}  {seconds / size * 1e6:>14.4f}")

In [ ]:
# Cost per compartment falls as the table grows: at small sizes Python
# object construction dominates, and from ~1e3 compartments the NumPy
# work does. That is what 'linear in the output table' looks like.
per_compartment = [seconds / size for size, seconds in build_rows]
assert per_compartment[-1] < per_compartment[0]

The smallest map is dominated by Python object construction; from a thousand
compartments upward the NumPy work dominates and the per-compartment cost
flattens.

## Querying

A cold query walks the selector tree once per node, each node costing one
`int8` pass over the table. A warm query is a dictionary lookup.

In [ ]:
pmap = cartesian((10, 10, 10, 10, 10))
assert pmap.size == 100_000

p0, p4 = pmap.properties[0], pmap.properties[4]
selector = p0["t0"] & p4["t0"]

cold = timed(lambda: pmap.copy().select(selector))
warm = timed(lambda: pmap.select(selector))

print(f"cold select over {pmap.size:,} compartments: {cold * 1e3:8.3f} ms")
print(f"warm select (cache hit):                    {warm * 1e3:8.3f} ms")
print(f"speedup: {cold / warm:,.0f}x")
assert warm < cold

## `isin` versus chained `or`

Selecting several traits of the same property is one node with `IsIn` and a tree
of `Or` nodes otherwise. The results are identical; the work is not.

In [ ]:
names = p0.traits[:5]

isin_sel = p0[tuple(names)]
or_sel = p0[names[0]]
for name in names[1:]:
    or_sel = or_sel | p0[name]

assert pmap.select(isin_sel).tolist() == pmap.select(or_sel).tolist()

isin_time = timed(lambda: pmap.copy().select(isin_sel))
or_time = timed(lambda: pmap.copy().select(or_sel))

print(f"isin (1 node)      : {isin_time * 1e3:7.3f} ms")
print(f"chained or (9 nodes): {or_time * 1e3:7.3f} ms")
print(f"ratio: {or_time / isin_time:.1f}x")

## Ragged versus rectangular

Raggedness costs nothing at query time — the third truth value is already in the
encoding — and it *saves* memory, because unstratified compartments are not
expanded.

In [ ]:
state = Property("state", ("S", "I", "R"))
age = Property("age", tuple(f"a{i}" for i in range(16)))
severity = Property("severity", ("mild", "severe"))

rectangular = PropertyMap.from_property(state).stratify(age).stratify(severity)
ragged = PropertyMap.from_property(state).stratify(age).stratify(severity, where=state["I"])

print(f"rectangular: {rectangular.size:>4} compartments, {rectangular.codes.nbytes:>5} bytes")
print(f"ragged     : {ragged.size:>4} compartments, {ragged.codes.nbytes:>5} bytes")

assert rectangular.size == 3 * 16 * 2 == 96
assert ragged.size == 16 + 16 * 2 + 16 == 64
assert ragged.codes.nbytes < rectangular.codes.nbytes

In [ ]:
rect_time = timed(lambda: rectangular.copy().select(state["I"] & severity["mild"]))
ragged_time = timed(lambda: ragged.copy().select(state["I"] & severity["mild"]))
print(f"rectangular query: {rect_time * 1e6:7.1f} us")
print(f"ragged query     : {ragged_time * 1e6:7.1f} us")

## Aggregation helpers

`partition` is one equality pass per trait. `group_by` sorts unique code rows, so
it is `O(n log n)` in the number of compartments that carry every requested
property.

In [ ]:
big = cartesian((10, 10, 10, 10))
prop0, prop1 = big.properties[0], big.properties[1]

partition_time = timed(lambda: big.partition(prop0))
group_time = timed(lambda: list(big.group_by(prop0, prop1)))

print(f"compartments      : {big.size:,}")
print(f"partition (1 axis): {partition_time * 1e3:7.3f} ms")
print(f"group_by (2 axes) : {group_time * 1e3:7.3f} ms")

assert sum(i.size for i in big.partition(prop0).values()) == big.size
assert sum(i.size for _, i in big.group_by(prop0, prop1)) == big.size

## What is not measured here

There is no measurement of model construction, integration or output
post-processing, because none of those exist. The `explorations/flows/` spike
contains a `lax.scan` Euler walk that has been checked against a NumPy loop, but
it is not part of the package and is not benchmarked — see {doc}`explorations`.